# Model Evaluation

This notebook evaluates the invoice NER model by:
1. Calling the localhost API endpoint
2. Comparing predictions against ground truth labels from `test_labels.json`
3. Computing evaluation metrics (accuracy, precision, recall, F1)

In [ ]:
import json
import requests
from pathlib import Path
from typing import Dict, List
from PIL import Image
import pandas as pd
from tqdm import tqdm
import base64
from io import BytesIO

## Configuration

In [ ]:
# API endpoint
API_URL = "http://localhost:8000"

# Data paths
TEST_JSON_PATH = "../data/SROIE2019/test/test.json"
TEST_LABELS_PATH = "../data/SROIE2019/test/test_labels.json"
TEST_IMG_DIR = "../data/SROIE2019/test/img"

# Output path
RESULTS_PATH = "../data/SROIE2019/test/evaluation_results.json"

## Health Check

Verify the API is running

In [ ]:
try:
    response = requests.get(f"{API_URL}/health", timeout=5)
    print(f"API Status: {response.json()}")
    if response.json().get('status') != 'healthy':
        print("⚠️ WARNING: API is not healthy!")
except Exception as e:
    print(f"❌ ERROR: Could not connect to API at {API_URL}")
    print(f"Error: {e}")
    print("\nMake sure the server is running with: uvicorn app:app --reload")

## Load Test Data

In [ ]:
# Load test dataset
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# Load ground truth labels
with open(TEST_LABELS_PATH, 'r', encoding='utf-8') as f:
    test_labels = json.load(f)

print(f"Loaded {len(test_data)} test samples")
print(f"Loaded {len(test_labels)} ground truth labels")
print(f"\nExample test data entry:")
print(f"  File: {test_data[0]['file']}")
print(f"  Words: {len(test_data[0]['words'])} words")
print(f"  Boxes: {len(test_data[0]['bboxes'])} boxes")
print(f"\nExample label:")
first_key = list(test_labels.keys())[0]
print(f"  {first_key}: {test_labels[first_key]}")

## Prediction Function

Function to call the API and get predictions

In [ ]:
def predict_via_api(image_path: str, words: List[str], boxes: List[List[int]]) -> Dict:
    """
    Call the localhost API to get predictions
    
    Args:
        image_path: Path to the image file
        words: List of OCR words
        boxes: List of bounding boxes
    
    Returns:
        Dictionary with prediction results
    """
    try:
        # Load image
        image = Image.open(image_path).convert('RGB')
        
        # Convert image to base64 for API call
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        
        # Prepare request payload
        payload = {
            "image": img_str,
            "words": words,
            "boxes": boxes
        }
        
        # Call API
        response = requests.post(
            f"{API_URL}/predict",
            json=payload,
            timeout=30
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            return {
                "error": f"API returned status {response.status_code}",
                "invoice_number": None
            }
    
    except Exception as e:
        return {
            "error": str(e),
            "invoice_number": None
        }

## Alternative: Direct Prediction (Without API)

If the API is not running, we can use the app.py functions directly

In [ ]:
# Import functions from app.py
import sys
sys.path.append('..')

from app import predict_invoice, load_model, postprocess_invoice_number, extract_invoice_heuristics
from PIL import Image

# Load model once
print("Loading model...")
load_model()
print("✅ Model loaded!")

def predict_direct(image_path: str, words: List[str], boxes: List[List[int]]) -> Dict:
    """
    Direct prediction using app.py functions (no API call)
    
    Args:
        image_path: Path to the image file
        words: List of OCR words
        boxes: List of bounding boxes
    
    Returns:
        Dictionary with prediction results
    """
    try:
        # Load image
        image = Image.open(image_path).convert('RGB')
        
        # Try heuristics first
        invoice_number, matched_indices = extract_invoice_heuristics(words, None)
        
        if invoice_number:
            # Heuristic found a match
            return {
                "invoice_number": invoice_number,
                "method": "heuristic"
            }
        else:
            # Fall back to model
            result = predict_invoice(image, words, boxes)
            invoice_number = result.get("invoice_number")
            
            # Apply postprocessing
            if invoice_number:
                invoice_number = postprocess_invoice_number(invoice_number)
            
            return {
                "invoice_number": invoice_number,
                "method": "model",
                "labels": result.get("labels"),
                "confidence_scores": result.get("confidence_scores")
            }
    
    except Exception as e:
        return {
            "error": str(e),
            "invoice_number": None
        }

## Run Evaluation

Predict on all test samples and compare with ground truth

In [ ]:
# Choose prediction method
USE_API = False  # Set to True to use API, False to use direct prediction

predict_fn = predict_via_api if USE_API else predict_direct

print(f"Using {'API' if USE_API else 'direct'} prediction method\n")

# Run predictions
results = []
errors = []

for item in tqdm(test_data, desc="Running predictions"):
    filename = item['file']
    image_path = Path(TEST_IMG_DIR) / filename
    
    # Skip if no ground truth label
    if filename not in test_labels:
        continue
    
    # Get ground truth
    ground_truth = test_labels[filename]
    
    # Get prediction
    prediction_result = predict_fn(
        str(image_path),
        item['words'],
        item['bboxes']
    )
    
    predicted = prediction_result.get('invoice_number')
    
    # Check for errors
    if 'error' in prediction_result:
        errors.append({
            'file': filename,
            'error': prediction_result['error']
        })
    
    # Store result
    results.append({
        'file': filename,
        'ground_truth': ground_truth,
        'predicted': predicted,
        'method': prediction_result.get('method', 'unknown'),
        'match': ground_truth == predicted if predicted else False
    })

print(f"\n✅ Completed {len(results)} predictions")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")

## Compute Metrics

In [ ]:
# Convert to DataFrame for analysis
df = pd.DataFrame(results)

# Overall accuracy
total = len(df)
correct = df['match'].sum()
accuracy = correct / total if total > 0 else 0

# Count predictions by method
method_counts = df['method'].value_counts()

# Count None predictions
none_predictions = df['predicted'].isna().sum()

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"\nTotal samples: {total}")
print(f"Correct predictions: {correct}")
print(f"Accuracy: {accuracy:.2%}")
print(f"\nPredictions by method:")
for method, count in method_counts.items():
    print(f"  {method}: {count} ({count/total:.1%})")
print(f"\nNo prediction (None): {none_predictions} ({none_predictions/total:.1%})")

if errors:
    print(f"\n⚠️ Errors: {len(errors)}")
    print("\nFirst 5 errors:")
    for err in errors[:5]:
        print(f"  {err['file']}: {err['error']}")

## Analyze Errors

Look at mismatches to understand failure modes

In [ ]:
# Get mismatches
mismatches = df[~df['match']].copy()

print(f"\nTotal mismatches: {len(mismatches)}\n")
print("First 20 mismatches:")
print("=" * 80)

for idx, row in mismatches.head(20).iterrows():
    print(f"File: {row['file']}")
    print(f"  Ground Truth: {row['ground_truth']}")
    print(f"  Predicted:    {row['predicted']}")
    print(f"  Method:       {row['method']}")
    print()

## Categorize Errors

In [ ]:
def categorize_error(row):
    """Categorize the type of error"""
    gt = str(row['ground_truth']) if row['ground_truth'] else ""
    pred = str(row['predicted']) if row['predicted'] else ""
    
    if not pred or pred == 'None':
        return 'No prediction'
    elif gt in pred or pred in gt:
        return 'Partial match'
    elif gt.replace(' ', '') == pred.replace(' ', ''):
        return 'Spacing difference'
    elif gt.replace('-', '') == pred.replace('-', ''):
        return 'Delimiter difference'
    else:
        return 'Complete mismatch'

mismatches['error_category'] = mismatches.apply(categorize_error, axis=1)

print("\nError categories:")
print(mismatches['error_category'].value_counts())
print()

# Show examples of each category
for category in mismatches['error_category'].unique():
    print(f"\n{category} examples:")
    examples = mismatches[mismatches['error_category'] == category].head(3)
    for _, row in examples.iterrows():
        print(f"  {row['file']}: '{row['ground_truth']}' vs '{row['predicted']}'")

## Save Results

In [ ]:
# Save detailed results
output = {
    'summary': {
        'total_samples': total,
        'correct_predictions': int(correct),
        'accuracy': float(accuracy),
        'method_counts': method_counts.to_dict(),
        'none_predictions': int(none_predictions)
    },
    'results': results,
    'errors': errors
}

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\n✅ Results saved to {RESULTS_PATH}")

# Also save mismatches to CSV for easy review
mismatches_csv = RESULTS_PATH.replace('.json', '_mismatches.csv')
mismatches.to_csv(mismatches_csv, index=False)
print(f"✅ Mismatches saved to {mismatches_csv}")

## Summary

Final evaluation summary

In [ ]:
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\n📊 Overall Accuracy: {accuracy:.2%}")
print(f"\n✅ Correct: {correct}/{total}")
print(f"❌ Incorrect: {total - correct}/{total}")
print(f"\n🎯 Heuristic predictions: {method_counts.get('heuristic', 0)}")
print(f"🤖 Model predictions: {method_counts.get('model', 0)}")
print(f"\n⚠️ No predictions: {none_predictions}")
print(f"\n💾 Results saved to: {RESULTS_PATH}")
print("=" * 60)